# Sookmyung Women's University — Deep Learning 2026: Lab 1
## Instructor: Prof. Joo Yong Sim

### Lab 0: Colab & PyTorch Basics
- PyTorch tensors and reduction operations
- Matrix operations and broadcasting
- GPU execution

### Lab 1: Optimization and ANN
- Optimization of linear regression
- ANN decision-boundary visualization
- Universal Approximation Theorem
- ANN for MNIST
- Cross-entropy loss vs. multi-class hinge loss
- L2 regularization and neuron contributions


# Lab 0: Colab & PyTorch Basics
PyTorch is an open-source machine-learning framework. In this course we use tensors, automatic differentiation, neural-network modules, and GPU acceleration throughout the practical sessions.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## Reduction operations
Reduction operations aggregate values over all or part of a tensor. Common examples are `sum`, `mean`, `min`, and `max`.


In [ ]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print('x =\n', x)
print('column sums:', x.sum(dim=0))
print('row means:', x.mean(dim=1))


### Exercise
Implement `zero_row_min(x)` so that the minimum value in each row is replaced by zero without using an explicit Python loop.


In [ ]:
def zero_row_min(x):
    y = x.clone()
    idx = x.argmin(dim=1)
    y[torch.arange(x.shape[0]), idx] = 0
    return y

x0 = torch.tensor([[10, 20, 30], [2, 5, 1]])
print(zero_row_min(x0))


## Matrix operations and broadcasting
PyTorch uses elementwise multiplication for `*` and matrix multiplication for `@`, `torch.mm`, or `torch.matmul`. Broadcasting allows compatible tensors of different shapes to participate in arithmetic without manually copying data.


In [ ]:
B, N, M, P = 2, 3, 5, 4
x = torch.randn(B, N, M)
y = torch.randn(B, M, P)
z = torch.bmm(x, y)
print('batched matrix multiplication shape:', z.shape)

a = torch.tensor([[1,2,3],[4,5,6],[7,8,9],[10,11,12]])
v = torch.tensor([1,0,1])
print(a + v)


## Running on GPU
In Google Colab, enable a GPU runtime when available. Tensors and models can be moved between CPU and GPU with `.to(device)`.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = torch.tensor([[1., 2.], [3., 4.]])
x = x.to(device)
print('device:', x.device)


# Lab 1: Optimization and ANN
## Optimization of Linear Regression
For mean-squared error, gradient descent repeatedly updates model parameters in the direction that reduces the loss. The following example uses PyTorch autograd to optimize a one-dimensional linear model.


In [ ]:
torch.manual_seed(0)
x = torch.linspace(-2, 2, 100).view(-1, 1)
y = 3.0 * x + 1.0 + 0.3 * torch.randn_like(x)

w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
lr = 0.05

history = []
for step in range(300):
    y_pred = w * x + b
    loss = ((y_pred - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    history.append(loss.item())

print('w =', w.item(), 'b =', b.item(), 'loss =', history[-1])
plt.plot(history)
plt.xlabel('step')
plt.ylabel('MSE loss')
plt.show()


## ANN decision boundary
A multilayer perceptron can learn nonlinear class boundaries by mapping the input through hidden nonlinear features.


In [ ]:
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(1)
N = 400
X = torch.rand(N, 2) * 2 - 1
target = ((X[:, 0] ** 2 + X[:, 1] ** 2) > 0.5).long()

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 8), nn.ReLU(),
            nn.Linear(8, 8), nn.ReLU(),
            nn.Linear(8, 2)
        )
    def forward(self, x):
        return self.net(x)

model = SmallMLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.02)
for epoch in range(300):
    logits = model(X)
    loss = criterion(logits, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(X).argmax(dim=1)
print('training accuracy:', (pred == target).float().mean().item())


# Universal Approximation Theorem
A feed-forward neural network with a hidden layer and suitable nonlinear activation can approximate a broad class of continuous functions. Here we approximate `sin(x)` with a one-hidden-layer ReLU network.


In [ ]:
x = torch.linspace(-np.pi, np.pi, 200).view(-1, 1)
y = torch.sin(x)

class Net(nn.Module):
    def __init__(self, hidden_size=50):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = Net(50)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()
for epoch in range(2000):
    y_pred = model(x)
    loss = criterion(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    predicted = model(x)
plt.plot(x.numpy(), y.numpy(), label='sin(x)')
plt.plot(x.numpy(), predicted.numpy(), label='ANN approximation')
plt.legend()
plt.show()


## Visualizing hidden-neuron contributions
The contribution of a hidden neuron can be examined by multiplying its activation by the weight that connects it to the output.


In [ ]:
with torch.no_grad():
    activations = torch.relu(model.fc1(x))
    weighted_activations = activations * model.fc2.weight.view(-1)

plt.figure(figsize=(10, 6))
for i in range(weighted_activations.shape[1]):
    plt.plot(x.numpy(), weighted_activations[:, i].numpy(), alpha=0.35)
plt.xlabel('x')
plt.ylabel('weighted activation')
plt.title('Hidden-neuron contributions')
plt.show()


# ANN for MNIST
Train a two-layer multilayer perceptron with 100 hidden neurons and ReLU activation on MNIST. Compare cross-entropy loss with a multi-class hinge (SVM) loss.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 100)
        self.fc2 = nn.Linear(100, 10)
    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


In [ ]:
def train_and_evaluate_model(model, loss_fn, optimizer, train_loader, test_loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)
            loss.backward()
            optimizer.step()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')

    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            pred = model(data).argmax(dim=1)
            correct += (pred == target).sum().item()
    print(f'Accuracy: {100.0 * correct / len(test_loader.dataset):.2f}%')


In [ ]:
print('Training with cross-entropy loss')
model_ce = MLP()
optimizer_ce = optim.SGD(model_ce.parameters(), lr=0.01, momentum=0.9)
train_and_evaluate_model(model_ce, nn.CrossEntropyLoss(), optimizer_ce, train_loader, test_loader)


## Multi-class hinge loss
For each training example, the hinge loss penalizes incorrect classes whose scores are too close to or larger than the correct-class score.


In [ ]:
class MultiClassHingeLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
    def forward(self, output, target):
        correct = output.gather(1, target.view(-1, 1))
        margins = torch.relu(output - correct + self.margin)
        margins.scatter_(1, target.view(-1, 1), 0.0)
        return margins.sum() / output.shape[0]

print('Training with hinge loss')
model_hinge = MLP()
optimizer_hinge = optim.SGD(model_hinge.parameters(), lr=0.01, momentum=0.9)
train_and_evaluate_model(model_hinge, MultiClassHingeLoss(), optimizer_hinge, train_loader, test_loader)


# L2 Regularization
L2 regularization (weight decay) discourages unnecessarily large model weights and can improve generalization. PyTorch optimizers expose it through the `weight_decay` argument.


In [ ]:
x = torch.linspace(-np.pi, np.pi, 200).view(-1, 1)
y = torch.sin(x)

def train_sine_model(weight_decay):
    model = Net(50)
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
    for epoch in range(2000):
        pred = model(x)
        loss = nn.functional.mse_loss(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

model_no_reg = train_sine_model(0.0)
model_with_reg = train_sine_model(0.01)
with torch.no_grad():
    p0 = model_no_reg(x)
    p1 = model_with_reg(x)
plt.plot(x.numpy(), y.numpy(), label='target')
plt.plot(x.numpy(), p0.numpy(), label='no regularization')
plt.plot(x.numpy(), p1.numpy(), label='L2 regularization')
plt.legend()
plt.show()
